# Clustering: Natural Resource Profiles

**Capstone Project -- Moody's Ratings**  
*Pipeline step 4 of 4: Exploratory classification of resource portfolios*

This notebook clusters countries by their per-capita natural resource production
profiles using PCA and K-Means. Three values of k (4, 5, 6) are compared.

**Input:** `NaturalResource.csv`, `Master.csv`, `sample_countries_final.csv`

---


## 0. Setup

In [1]:
import os
os.makedirs("intermediary", exist_ok=True)
os.makedirs("Graphics/NB4", exist_ok=True)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# ── Color palettes (one per k) ──
# Designed to be distinguishable on both scatter plots and choropleth maps.
# Ordered so that the first colour (assigned to the highest-PC1 cluster,
# typically oil-dominant) is always the same warm tone across k values.
PALETTES = {
    4: ["#E63946", "#457B9D", "#2A9D8F", "#A8DADC"],
    5: ["#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#A8DADC"],
    6: ["#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#264653", "#A8DADC"],
}


## 1. Load Data and Define Sample

The sample is read from `sample_countries_final.csv`, produced by the
selection pipeline in NB3 (`sample_selection.py`).


In [2]:
nr = pd.read_csv("intermediary/NaturalResource.csv")

# Read sample from NB3 pipeline output
include_list = pd.read_csv(
    "intermediary/sample_countries_final.csv"
)["Country Code"].tolist()

nr_sample = nr[nr["Country Code"].isin(include_list)]
print(f"NR data: {nr_sample.shape[0]:,} rows")
print(f"Countries: {nr_sample['Country Code'].nunique()}")
print(f"Years: {nr_sample['Year'].min()}-{nr_sample['Year'].max()}")
print(f"Resources: {nr_sample['Resource'].nunique()}")


NR data: 7,262 rows
Countries: 47
Years: 1995.0-2021.0
Resources: 21


## 2. Clustering Pipeline

The clustering function encapsulates the full pipeline: pivot to per-capita
production values, log-transform, reduce to two principal components, then
apply K-Means. Cluster labels are assigned automatically by ranking centroids
along PC1 and PC2 (which load on hydrocarbons and minerals respectively).

For k=4 the labels follow the original four-category scheme. For k=5 and k=6
the additional clusters are labeled by their centroid position relative to
the existing groups.


In [3]:
def run_clustering(nr_data, year_filter=None, agg_years=None, n_clusters=4, random_state=42):
    """
    Full clustering pipeline: pivot -> per capita -> log1p -> PCA(2) -> KMeans(k).
    Returns pca_df, pca_model, feature_cols.
    """
    df = nr_data.copy()

    if year_filter is not None:
        df = df[df["Year"] == year_filter]
    elif agg_years is not None:
        df = df[df["Year"].isin(agg_years)]

    df_pivot = df.pivot_table(
        index=["Country", "Country Code", "Year", "Population"],
        columns="Resource",
        values="Production_TotalValue",
    ).reset_index()

    resource_cols = df_pivot.columns.difference(
        ["Country", "Country Code", "Year", "Population"]
    )
    df_pivot[resource_cols] = df_pivot[resource_cols].div(
        df_pivot["Population"], axis=0
    )
    df_pivot.drop(columns="Population", inplace=True)
    df_pivot = df_pivot.fillna(0)

    df_latest = (
        df_pivot.sort_values("Year", ascending=True)
        .groupby(["Country", "Country Code"])
        .first()
        .reset_index()
    )

    feature_cols = [c for c in df_latest.columns
                    if c not in ["Country", "Country Code", "Year"]]

    X = df_latest[feature_cols].fillna(0)
    X_log = np.log1p(X)

    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(X_log)

    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    clusters = kmeans.fit_predict(pca_components)

    pca_df = pd.DataFrame({
        "Country": df_latest["Country"],
        "Country Code": df_latest["Country Code"],
        "Year": df_latest["Year"],
        "PC1": pca_components[:, 0],
        "PC2": pca_components[:, 1],
        "Cluster": clusters,
    })

    # ── Auto-label clusters from centroids ──
    centroids = kmeans.cluster_centers_
    pc1_rank = list(np.argsort(-centroids[:, 0]))  # highest PC1 first
    pc2_rank = list(np.argsort(-centroids[:, 1]))  # highest PC2 first

    label_map = {}
    labeled = set()

    # Labels derived from actual per-capita production composition:
    # Highest PC1 -> Petrostates (highest per-capita hydrocarbon production)
    # Highest PC2 (remaining) -> Diversified Producers (oil + coal + metals mix)
    # Next by PC1 -> Oil Exporters (oil-dominant, mid-scale per capita)
    # Lowest -> Hard Mineral Exporters (copper/gold/coal, no oil)

    # 1. Highest PC1 -> Petrostates
    oil_id = pc1_rank[0]
    label_map[oil_id] = "Petrostates"
    labeled.add(oil_id)

    # 2. Highest PC2 not yet labeled -> Diversified Producers
    mineral_id = next(c for c in pc2_rank if c not in labeled)
    label_map[mineral_id] = "Diversified Producers"
    labeled.add(mineral_id)

    # 3. Remaining by PC1 rank
    remaining = [c for c in pc1_rank if c not in labeled]
    if len(remaining) == 2:
        # k=4
        label_map[remaining[0]] = "Oil Exporters"
        label_map[remaining[1]] = "Hard Mineral Exporters"
    elif len(remaining) == 3:
        # k=5
        label_map[remaining[0]] = "Oil Exporters"
        mid = remaining[1]
        low = remaining[2]
        if centroids[mid, 1] > centroids[low, 1]:
            label_map[mid] = "Mineral-Rich Developing"
            label_map[low] = "Hard Mineral Exporters"
        else:
            label_map[mid] = "Low-Intensity Producers"
            label_map[low] = "Hard Mineral Exporters"
    elif len(remaining) >= 4:
        # k=6+
        label_map[remaining[0]] = "Oil Exporters"
        for j, cid in enumerate(remaining[1:], 1):
            if centroids[cid, 1] > np.median(centroids[:, 1]):
                label_map[cid] = f"Mineral-Rich Developing ({j})"
            else:
                label_map[cid] = f"Low-Intensity Producers ({j})"
    else:
        for cid in remaining:
            label_map[cid] = f"Cluster {cid}"

    pca_df["ClusterLabels"] = pca_df["Cluster"].map(label_map)

    sil = silhouette_score(pca_components, clusters)
    print(f"k={n_clusters}, Silhouette: {sil:.3f}")
    for cid in sorted(pca_df["Cluster"].unique()):
        n = (pca_df["Cluster"] == cid).sum()
        print(f"  {label_map[cid]}: {n} countries")

    return pca_df, pca, feature_cols


## 3. Validate k with Silhouette Analysis

Before running the final clustering, we check silhouette scores for k=2 through
k=8. The three values used in the analysis (k=4, 5, 6) are highlighted.


In [4]:
nr_1995 = nr_sample[nr_sample["Year"] == 1995].copy()

df_pivot_val = nr_1995.pivot_table(
    index=["Country", "Country Code", "Year", "Population"],
    columns="Resource",
    values="Production_TotalValue",
).reset_index()

resource_cols_val = df_pivot_val.columns.difference(
    ["Country", "Country Code", "Year", "Population"]
)
df_pivot_val[resource_cols_val] = df_pivot_val[resource_cols_val].div(
    df_pivot_val["Population"], axis=0
)
df_pivot_val = df_pivot_val.fillna(0)

feat_cols_val = [c for c in df_pivot_val.columns
                 if c not in ["Country", "Country Code", "Year", "Population"]]
X_val = np.log1p(df_pivot_val[feat_cols_val].fillna(0))

pca_val = PCA(n_components=2)
X_pca_val = pca_val.fit_transform(X_val)

k_range = range(2, 9)
sil_scores = []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca_val)
    sil_scores.append(silhouette_score(X_pca_val, labels))

fig_val = go.Figure()
fig_val.add_trace(go.Scatter(
    x=list(k_range), y=sil_scores,
    mode="lines+markers", name="Silhouette Score",
    marker=dict(size=10, color=["#E63946" if k in (4,5,6) else "#457B9D" for k in k_range]),
))
for k_sel in [4, 5, 6]:
    fig_val.add_vline(x=k_sel, line_dash="dash", line_color="grey", line_width=0.8)
fig_val.update_layout(
    title="Silhouette Score by Number of Clusters (1995 data)",
    xaxis_title="k", yaxis_title="Silhouette Score",
    width=700, height=400,
)
fig_val.show()

print("\nSilhouette scores:")
for k, s in zip(k_range, sil_scores):
    marker = " <-- selected" if k in (4, 5, 6) else ""
    print(f"  k={k}: {s:.3f}{marker}")



Silhouette scores:
  k=2: 0.392
  k=3: 0.396
  k=4: 0.399 <-- selected
  k=5: 0.472 <-- selected
  k=6: 0.498 <-- selected
  k=7: 0.487
  k=8: 0.501


## 4. Run Clustering for k=4, 5, 6

Each k is run on three time slices:
- **1995:** single-year snapshot at the start of the panel
- **2019:** single-year snapshot at the end of the panel
- **Aggregated (1995, 1999, 2005):** earliest available year per country


In [5]:
results = {}     # keyed by (k, time_variant)
pca_models = {}  # keyed by (k, time_variant)
feat_cols = {}   # keyed by (k, time_variant)

K_VALUES = [4, 5, 6]
TIME_VARIANTS = [
    ("1995", dict(year_filter=1995)),
    ("2019", dict(year_filter=2019)),
    ("agg",  dict(agg_years=[1995, 1999, 2005])),
]

for k in K_VALUES:
    print("=" * 60)
    print(f"k = {k}")
    print("=" * 60)
    for label, kwargs in TIME_VARIANTS:
        print(f"\n--- {label} ---")
        pca_df, pca_mod, feats = run_clustering(
            nr_sample, n_clusters=k, **kwargs
        )
        results[(k, label)] = pca_df
        pca_models[(k, label)] = pca_mod
        feat_cols[(k, label)] = feats
    print()


k = 4

--- 1995 ---
k=4, Silhouette: 0.399
  Oil Exporters: 16 countries
  Hard Mineral Exporters: 13 countries
  Diversified Producers: 10 countries
  Petrostates: 8 countries

--- 2019 ---
k=4, Silhouette: 0.428
  Oil Exporters: 18 countries
  Diversified Producers: 7 countries
  Hard Mineral Exporters: 10 countries
  Petrostates: 12 countries

--- agg ---
k=4, Silhouette: 0.399
  Oil Exporters: 16 countries
  Hard Mineral Exporters: 13 countries
  Diversified Producers: 10 countries
  Petrostates: 8 countries

k = 5

--- 1995 ---
k=5, Silhouette: 0.472
  Oil Exporters: 11 countries
  Petrostates: 11 countries
  Hard Mineral Exporters: 6 countries
  Low-Intensity Producers: 11 countries
  Diversified Producers: 8 countries

--- 2019 ---
k=5, Silhouette: 0.457
  Oil Exporters: 18 countries
  Diversified Producers: 7 countries
  Petrostates: 11 countries
  Low-Intensity Producers: 8 countries
  Hard Mineral Exporters: 3 countries

--- agg ---
k=5, Silhouette: 0.472
  Oil Exporters: 11 

k=6, Silhouette: 0.498
  Diversified Producers: 2 countries
  Oil Exporters: 11 countries
  Petrostates: 11 countries
  Mineral-Rich Developing (3): 5 countries
  Mineral-Rich Developing (1): 7 countries
  Low-Intensity Producers (2): 11 countries

--- 2019 ---
k=6, Silhouette: 0.484
  Mineral-Rich Developing (3): 3 countries
  Low-Intensity Producers (1): 15 countries
  Oil Exporters: 8 countries
  Diversified Producers: 3 countries
  Petrostates: 10 countries
  Low-Intensity Producers (2): 8 countries

--- agg ---
k=6, Silhouette: 0.498
  Diversified Producers: 2 countries
  Oil Exporters: 11 countries
  Petrostates: 11 countries
  Mineral-Rich Developing (3): 5 countries
  Mineral-Rich Developing (1): 7 countries
  Low-Intensity Producers (2): 11 countries



## 5. PCA Loadings Analysis

The loadings reveal which resources drive each principal component. PC1 is expected to capture hydrocarbon abundance (oil, natural gas), while PC2 should capture mineral production (copper, gold, zinc, etc.). This interpretation is central to the cluster labelling logic.

In [6]:
# Use k=4, 1995 model for loadings analysis (consistent with report)
pca_model_ref = pca_models[(4, "1995")]
feat_ref = feat_cols[(4, "1995")]

loadings = pd.DataFrame(
    pca_model_ref.components_.T,
    columns=["PC1", "PC2"],
    index=feat_ref,
)

print("PCA Explained Variance (1995):")
for i, var in enumerate(pca_model_ref.explained_variance_ratio_):
    cum = pca_model_ref.explained_variance_ratio_[:i + 1].sum()
    print(f"  PC{i+1}: {var*100:.1f}% (cumulative: {cum*100:.1f}%)")

top_features = loadings.abs().sum(axis=1).nlargest(20).index

fig_load = px.imshow(
    loadings.loc[top_features].T,
    labels=dict(x="Resource", y="Principal Component", color="Loading"),
    title="PCA Factor Loadings (Top 20 Resources, 1995)",
    color_continuous_scale="RdBu_r",
    aspect="auto", zmin=-1, zmax=1,
)
fig_load.update_layout(width=1100, height=350)
fig_load.show()

print("\nTop loadings by component:")
for pc in ["PC1", "PC2"]:
    print(f"\n{pc}:")
    sorted_l = loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index)
    for feat, val in sorted_l.head(8).items():
        print(f"  {feat:35s} {val:+.4f}")


PCA Explained Variance (1995):
  PC1: 41.3% (cumulative: 41.3%)
  PC2: 20.8% (cumulative: 62.1%)



Top loadings by component:

PC1:
  Oil                                 +0.7552
  Natural Gas                         +0.5500
  Copper                              -0.1905
  Gold                                -0.1789
  Coal                                -0.1781
  Aluminium                           +0.1297
  Bauxite                             -0.0583
  Silver                              -0.0490

PC2:
  Coal                                +0.5497
  Copper                              +0.4694
  Natural Gas                         +0.3757
  Gold                                +0.3369
  Iron ore                            +0.3017
  Aluminium                           +0.2383
  Zinc                                +0.1828
  Silver                              +0.1596


## 6. Biplots: PCA Space with Cluster Assignments

One biplot per k value (using the 1995 snapshot), showing how the PCA space
is partitioned as the number of clusters increases.


In [7]:
def create_biplot(pca_df, pca_model, feature_cols, k, title_suffix=""):
    """Create PCA biplot with cluster colours and loading arrows."""

    loadings_plot = pca_model.components_.T * np.sqrt(pca_model.explained_variance_)
    loadings_df = pd.DataFrame(loadings_plot[:, :2], columns=["PC1", "PC2"], index=feature_cols)
    scale_factor = 2.5
    loadings_scaled = loadings_df * scale_factor

    importance = loadings_df.abs().sum(axis=1)
    top_n = min(15, len(feature_cols))
    top_feats = importance.nlargest(top_n).index

    palette = PALETTES.get(k, px.colors.qualitative.Bold[:k])
    color_map = {}
    for cid in sorted(pca_df["Cluster"].unique()):
        lbl = pca_df[pca_df["Cluster"] == cid]["ClusterLabels"].iloc[0]
        color_map[lbl] = palette[cid % len(palette)]

    fig = px.scatter(
        pca_df, x="PC1", y="PC2",
        color="ClusterLabels",
        hover_data=["Country", "Country Code", "Year"],
        title=f"PCA Biplot (k={k}){title_suffix}",
        color_discrete_map=color_map,
    )

    for feat in top_feats:
        fig.add_annotation(
            x=loadings_scaled.loc[feat, "PC1"],
            y=loadings_scaled.loc[feat, "PC2"],
            ax=0, ay=0, xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="black",
        )
        fig.add_annotation(
            x=loadings_scaled.loc[feat, "PC1"] * 1.15,
            y=loadings_scaled.loc[feat, "PC2"] * 1.15,
            text=feat, showarrow=False,
            font=dict(size=9, color="black"),
        )

    var1 = pca_model.explained_variance_ratio_[0] * 100
    var2 = pca_model.explained_variance_ratio_[1] * 100
    fig.update_layout(
        width=1000, height=700,
        xaxis_title=f"PC1 ({var1:.1f}%)",
        yaxis_title=f"PC2 ({var2:.1f}%)",
    )
    return fig


for k in K_VALUES:
    fig = create_biplot(
        results[(k, "1995")], pca_models[(k, "1995")],
        feat_cols[(k, "1995")], k, f" -- 1995"
    )
    fig.show()
    fig.write_html(f"Graphics/NB4/biplot_k{k}_1995.html")


## 7. Choropleth Maps

One map per k value (using the 1995 snapshot). Countries producing more than
15% of a resource's global output are flagged with a red border.


In [8]:
def create_cluster_map(pca_df, nr_data, k, cluster_names_map=None, dominance_threshold=15.0):
    """Choropleth map with red borders for major global producers."""

    palette = PALETTES.get(k, px.colors.qualitative.Bold[:k])

    if cluster_names_map is None:
        cluster_names_map = dict(
            zip(pca_df["Cluster"].unique(), pca_df["ClusterLabels"].unique())
        )

    # ── Global production shares ──
    df_total = nr_data.pivot_table(
        index=["Country", "Country Code"],
        columns="Resource",
        values="Production_TotalValue",
        aggfunc="sum",
    ).reset_index().fillna(0)

    prod_cols = [c for c in df_total.columns if c not in ["Country", "Country Code"]]
    for col in prod_cols:
        total = df_total[col].sum()
        if total > 0:
            df_total[f"{col}_Share"] = (df_total[col] / total) * 100

    share_cols = [c for c in df_total.columns if c.endswith("_Share")]
    df_map = pca_df.merge(df_total[["Country Code"] + share_cols], on="Country Code", how="left")
    df_map["Is_Dominant"] = (df_map[share_cols] >= dominance_threshold).any(axis=1)
    df_map["Dominant_Resources"] = df_map.apply(
        lambda row: [sc.replace("_Share", "") for sc in share_cols
                     if row.get(sc, 0) >= dominance_threshold], axis=1,
    )

    def make_hover(row):
        lines = [f"<b>{row['Country']}</b>", f"Cluster: {row['ClusterLabels']}"]
        vals = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        if vals:
            lines.append("<br>Top Resources:")
            for res, v in vals[:3]:
                if v > 1e9:
                    lines.append(f"  {res}: ${v/1e9:.1f}B")
                elif v > 1e6:
                    lines.append(f"  {res}: ${v/1e6:.0f}M")
                else:
                    lines.append(f"  {res}: ${v:,.0f}")
        return "<br>".join(lines)

    df_map["hover_text"] = df_map.apply(make_hover, axis=1)

    fig = go.Figure()

    for cid in sorted(df_map["Cluster"].unique()):
        lbl = cluster_names_map.get(cid, f"Cluster {cid}")
        color = palette[cid % len(palette)]

        sub = df_map[(df_map["Cluster"] == cid) & (~df_map["Is_Dominant"])]
        if len(sub) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub["Country Code"], z=[cid]*len(sub),
                colorscale=[[0, color], [1, color]], showscale=False,
                customdata=sub["hover_text"].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{lbl} ({len(sub)})",
                marker=dict(line=dict(color="white", width=0.5)),
            ))

        sub_d = df_map[(df_map["Cluster"] == cid) & (df_map["Is_Dominant"])]
        if len(sub_d) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub_d["Country Code"], z=[cid]*len(sub_d),
                colorscale=[[0, color], [1, color]], showscale=False,
                customdata=sub_d["hover_text"].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{lbl} (major producer, {len(sub_d)})",
                marker=dict(line=dict(color="red", width=1.5)),
            ))

    fig.update_geos(
        projection_type="natural earth",
        showcountries=True, countrycolor="lightgray",
        showcoastlines=True, coastlinecolor="lightgray",
        showland=True, landcolor="whitesmoke",
        showocean=True, oceancolor="aliceblue",
    )
    fig.update_layout(
        title=dict(
            text=f"Natural Resource Clusters, k={k} (per capita production value)<br>"
                 "<sup>Red border = >15% of a resource global production</sup>",
            x=0.45, font=dict(size=14),
        ),
        width=1100, height=550,
        margin=dict(l=10, r=150, t=60, b=10),
        legend=dict(x=1.01, y=0.3, font=dict(size=10)),
    )
    return fig


nr_1995_full = nr_sample[nr_sample["Year"] == 1995]
for k in K_VALUES:
    fig_map = create_cluster_map(results[(k, "1995")], nr_1995_full, k)
    fig_map.show()
    fig_map.write_html(f"Graphics/NB4/map_k{k}_1995.html")


In [9]:
# ── TEMPORARY: dump cluster assignments + production profiles ──
# Remove this cell after use.
import json as _json

nr_1995 = nr_sample[nr_sample['Year'] == 1995].copy()

_dump = {}
for k in K_VALUES:
    df = results[(k, '1995')]

    # Merge NR production data
    prod = nr_1995.pivot_table(
        index='Country Code', columns='Resource',
        values='Production_TotalValue', aggfunc='sum',
    ).fillna(0)

    merged = df.merge(prod, on='Country Code', how='left')
    resource_cols = [c for c in prod.columns]

    # Per-cluster average production
    cluster_profiles = []
    for cid in sorted(df['Cluster'].unique()):
        sub = merged[merged['Cluster'] == cid]
        lbl = sub['ClusterLabels'].iloc[0]
        avg = sub[resource_cols].mean()
        top5 = avg.nlargest(5)
        countries = sorted(sub['Country Code'].tolist())
        cluster_profiles.append({
            'cl': int(cid),
            'lb': lbl,
            'n': len(sub),
            'countries': countries,
            'top5': {r: round(float(v), 0) for r, v in top5.items()},
        })

    _dump[str(k)] = cluster_profiles

print(_json.dumps(_dump, indent=2))


{
  "4": [
    {
      "cl": 0,
      "lb": "Oil Exporters",
      "n": 16,
      "countries": [
        "AGO",
        "ARG",
        "AZE",
        "BOL",
        "CMR",
        "COG",
        "COL",
        "ECU",
        "EGY",
        "EST",
        "GAB",
        "IRQ",
        "MYS",
        "NGA",
        "TUN",
        "YEM"
      ],
      "top5": {
        "Oil": 3034353116.0,
        "Natural Gas": 331806246.0,
        "Coal": 64734624.0,
        "Aluminium": 57539321.0,
        "Gold": 36842725.0
      }
    },
    {
      "cl": 1,
      "lb": "Hard Mineral Exporters",
      "n": 13,
      "countries": [
        "ALB",
        "BIH",
        "BWA",
        "COD",
        "DOM",
        "GHA",
        "GIN",
        "JAM",
        "MMR",
        "MRT",
        "PAK",
        "VNM",
        "ZWE"
      ],
      "top5": {
        "Oil": 127429618.0,
        "Gold": 84808369.0,
        "Natural Gas": 70215169.0,
        "Coal": 55734307.0,
        "Bauxite": 52759832.0
      }


## 8. ECI vs GDP Evolution (Rosling Chart)

This animated chart tracks how each country's Economic Complexity Index evolved
against log GDP per capita over the panel period. Cluster assignments are fixed
at 1995 values. Bubble size reflects per-capita production value.

The Rosling chart uses k=4 (the primary specification for the report).


In [10]:
master = pd.read_csv("intermediary/Master.csv")
master = master[master["Country Code"].isin(include_list)]

# Use k=4, aggregated for Rosling chart
pca_agg_k4 = results[(4, "agg")]

master = pd.merge(
    master,
    pca_agg_k4[["Country Code", "Cluster", "ClusterLabels"]],
    on="Country Code",
    how="left",
)

palette_k4 = PALETTES[4]
CLUSTER_COLORS = {cid: palette_k4[cid % len(palette_k4)]
                  for cid in sorted(pca_agg_k4["Cluster"].unique())}
CLUSTER_NAMES = dict(zip(pca_agg_k4["Cluster"], pca_agg_k4["ClusterLabels"]))


In [11]:
def create_rosling_chart(df, cluster_colors, cluster_names, arrow_opacity=0.5, arrow_width=2):
    """Animated ECI vs log(GDP pc) chart with trajectory arrows from 1995."""

    data = df.copy()
    data["Log GDP per capita"] = np.log(data["GDP per capita (constant prices, PPP)"])
    data["Production_Per_Capita"] = data["Total_Production_Value"] / data["Population"]

    # Fix cluster to 1995 value
    c1995 = data[data["Year"] == 1995][["Country Code", "Cluster"]].copy()
    c1995 = c1995.rename(columns={"Cluster": "Cluster_1995"})
    data = data.merge(c1995, on="Country Code", how="left")
    data = data.dropna(subset=["Cluster_1995", "Log GDP per capita",
                                "Economic Complexity Index", "Production_Per_Capita"])
    data["Cluster_1995"] = data["Cluster_1995"].astype(int)

    # Bubble size
    data["Bubble_Size"] = np.sqrt(data["Production_Per_Capita"])
    mn, mx = data["Bubble_Size"].min(), data["Bubble_Size"].max()
    data["Bubble_Size_Scaled"] = 8 + (data["Bubble_Size"] - mn) / (mx - mn) * 42

    data = data.sort_values(["Year", "Country Code"])
    years = sorted(data["Year"].unique())
    countries_list = data["Country Code"].unique()
    clusters = sorted(data["Cluster_1995"].unique())

    # Build per-country data
    cdata = {}
    for code in countries_list:
        cdf = data[data["Country Code"] == code].sort_values("Year")
        origin = cdf[cdf["Year"] == 1995]
        if len(origin) == 0:
            continue
        cdata[code] = {
            "years": cdf["Year"].values,
            "x": cdf["Log GDP per capita"].values,
            "y": cdf["Economic Complexity Index"].values,
            "x0": origin["Log GDP per capita"].values[0],
            "y0": origin["Economic Complexity Index"].values[0],
            "size": cdf["Bubble_Size_Scaled"].values,
            "name": cdf["Country Name"].iloc[0],
            "cluster": cdf["Cluster_1995"].iloc[0],
            "prod_pc": cdf["Production_Per_Capita"].values,
        }

    valid_countries = list(cdata.keys())
    first_year = years[0]

    fig = go.Figure()

    for cl in clusters:
        cc = [c for c in valid_countries if cdata[c]["cluster"] == cl]
        color = cluster_colors.get(cl, "#999999")

        for code in cc:
            cd = cdata[code]
            idx = np.where(cd["years"] == first_year)[0]
            xc = cd["x"][idx[0]] if len(idx) > 0 else cd["x0"]
            yc = cd["y"][idx[0]] if len(idx) > 0 else cd["y0"]
            fig.add_trace(go.Scatter(
                x=[cd["x0"], xc], y=[cd["y0"], yc],
                mode="lines", line=dict(color=color, width=arrow_width),
                opacity=arrow_opacity, legendgroup=f"cl_{cl}", showlegend=False, hoverinfo="skip",
            ))

        for code in cc:
            cd = cdata[code]
            idx = np.where(cd["years"] == first_year)[0]
            if len(idx) > 0:
                i = idx[0]
                xv, yv, sv, pv = [cd["x"][i]], [cd["y"][i]], cd["size"][i], cd["prod_pc"][i]
            else:
                xv, yv, sv, pv = [cd["x0"]], [cd["y0"]], 15, 0
            fig.add_trace(go.Scatter(
                x=xv, y=yv, mode="markers+text",
                marker=dict(size=sv, color=color, opacity=0.85, line=dict(width=1.5, color="white")),
                text=[code], textposition="top center", textfont=dict(size=8, color="black"),
                name=cluster_names.get(cl, f"Cluster {cl}"),
                legendgroup=f"cl_{cl}", showlegend=(code == cc[0]),
                customdata=[[cd["name"], pv, first_year]],
                hovertemplate="<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>"
                              "ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>"
                              "Year: %{customdata[2]}<extra></extra>",
            ))

        for code in cc:
            cd = cdata[code]
            fig.add_trace(go.Scatter(
                x=[cd["x0"]], y=[cd["y0"]], mode="markers",
                marker=dict(size=5, color=color, opacity=0.6, symbol="circle"),
                legendgroup=f"cl_{cl}", showlegend=False, hoverinfo="skip",
            ))

    # Frames for animation
    frames = []
    for year in years:
        fd = []
        for cl in clusters:
            cc = [c for c in valid_countries if cdata[c]["cluster"] == cl]
            color = cluster_colors.get(cl, "#999999")
            for code in cc:
                cd = cdata[code]
                idx = np.where(cd["years"] == year)[0]
                if len(idx) > 0:
                    xc, yc = cd["x"][idx[0]], cd["y"][idx[0]]
                else:
                    mask = cd["years"] <= year
                    li = np.where(mask)[0][-1] if mask.any() else 0
                    xc, yc = cd["x"][li], cd["y"][li]
                fd.append(go.Scatter(x=[cd["x0"], xc], y=[cd["y0"], yc],
                                     mode="lines", line=dict(color=color, width=arrow_width), opacity=arrow_opacity))
            for code in cc:
                cd = cdata[code]
                idx = np.where(cd["years"] == year)[0]
                if len(idx) > 0:
                    i = idx[0]
                    xv, yv, sv, pv = [cd["x"][i]], [cd["y"][i]], cd["size"][i], cd["prod_pc"][i]
                else:
                    mask = cd["years"] <= year
                    if mask.any():
                        li = np.where(mask)[0][-1]
                        xv, yv, sv, pv = [cd["x"][li]], [cd["y"][li]], cd["size"][li], cd["prod_pc"][li]
                    else:
                        xv, yv, sv, pv = [cd["x0"]], [cd["y0"]], 15, 0
                fd.append(go.Scatter(
                    x=xv, y=yv, mode="markers+text",
                    marker=dict(size=sv, color=color, opacity=0.85, line=dict(width=1.5, color="white")),
                    text=[code], textposition="top center", textfont=dict(size=8),
                    customdata=[[cd["name"], pv, year]],
                    hovertemplate="<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>"
                                  "ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>"
                                  "Year: %{customdata[2]}<extra></extra>",
                ))
            for code in cc:
                cd = cdata[code]
                fd.append(go.Scatter(x=[cd["x0"]], y=[cd["y0"]], mode="markers",
                                     marker=dict(size=5, color=color, opacity=0.6, symbol="circle")))
        frames.append(go.Frame(data=fd, name=str(year)))

    fig.frames = frames

    eci_vals = data["Economic Complexity Index"]
    x_vals = data["Log GDP per capita"]
    fig.update_layout(
        title=dict(text="Evolution of Economic Complexity vs Income<br>"
                        "<sup>Bubble size = Production per Capita</sup>", x=0.5),
        xaxis=dict(range=[x_vals.min()-0.2, x_vals.max()+0.2], title="Log GDP per capita (PPP)"),
        yaxis=dict(range=[eci_vals.min()-0.5, eci_vals.max()+0.5], title="Economic Complexity Index"),
        plot_bgcolor="white", width=850, height=650,
        legend=dict(title="Resource Profile (1995)", x=1.02, y=0.99),
        updatemenus=[dict(
            type="buttons", showactive=True, x=1.0, y=-0.02,
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, dict(frame=dict(duration=500, redraw=True), transition=dict(duration=300))]),
                dict(label="Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            active=0, len=0.85, x=0.05, y=-0.12,
            currentvalue=dict(prefix="Year: ", font=dict(size=14)),
            steps=[dict(args=[[str(y)], dict(frame=dict(duration=300, redraw=True), mode="immediate")],
                        method="animate", label=str(y)) for y in years],
        )],
    )
    return fig


fig_rosling = create_rosling_chart(master, CLUSTER_COLORS, CLUSTER_NAMES)
fig_rosling.show()

## 9. Add Hover Text and Export

The cluster CSVs include hover text with top resources for each country, matching the format used in the original analysis files.

In [12]:
def add_hover_text(pca_df, nr_data):
    """Add hover text with top resources to cluster DataFrame."""
    totals = nr_data.pivot_table(
        index=["Country", "Country Code"],
        columns="Resource",
        values="Production_TotalValue",
        aggfunc="sum",
    ).reset_index().fillna(0)

    prod_cols = [c for c in totals.columns if c not in ["Country", "Country Code"]]
    df = pca_df.merge(totals, on="Country Code", how="left", suffixes=("", "_nr"))

    def make_ht(row):
        lines = [f"<b>{row['Country']}</b>",
                 f"Cluster: {row['ClusterLabels']}"]
        lines.append("<br>Top Resources:")
        vals = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        for res, v in vals[:3]:
            lines.append(f"  {res}: ${v:,.0f}")
        return "<br>".join(lines)

    df["hover_text"] = df.apply(make_ht, axis=1)
    return df[["Country", "Country Code", "Year", "PC1", "PC2",
               "Cluster", "ClusterLabels", "hover_text"]]


# Export cluster CSVs for all k values and time variants
for k in K_VALUES:
    for label, kwargs in TIME_VARIANTS:
        if label == "agg":
            nr_sub = nr_sample[nr_sample["Year"].isin([1995, 1999, 2005])]
        else:
            nr_sub = nr_sample[nr_sample["Year"] == int(label)]
        out = add_hover_text(results[(k, label)], nr_sub)
        out_path = f"intermediary/clusters_k{k}_{label}.csv"
        out.to_csv(out_path, index=False)
        print(f"Saved {out_path}: {len(out)} countries")

print("\nDone. All cluster CSVs exported.")


Saved intermediary/clusters_k4_1995.csv: 47 countries
Saved intermediary/clusters_k4_2019.csv: 47 countries
Saved intermediary/clusters_k4_agg.csv: 47 countries
Saved intermediary/clusters_k5_1995.csv: 47 countries
Saved intermediary/clusters_k5_2019.csv: 47 countries
Saved intermediary/clusters_k5_agg.csv: 47 countries
Saved intermediary/clusters_k6_1995.csv: 47 countries
Saved intermediary/clusters_k6_2019.csv: 47 countries
Saved intermediary/clusters_k6_agg.csv: 47 countries

Done. All cluster CSVs exported.


---

## Summary

| Component | Choice | Rationale |
|-----------|--------|-----------|
| **Normalisation** | Per-capita (production value / population) | Controls for country size |
| **Transform** | log(1+x) | Compresses skewed production values |
| **Reduction** | PCA (2 components) | Captures hydrocarbon vs mineral axis |
| **Clustering** | K-Means, k = 4, 5, 6 | Silhouette-validated; k=4 is primary |
| **Time slices** | 1995, 2019, aggregated | Tests stability of assignments |


In [13]:
# ── NB4 SUMMARY ──
print("=" * 70)
print("NB4: CLUSTERING SUMMARY")
print("=" * 70)

pca_ref = pca_models[(4, "1995")]
print(f"\nPCA variance explained (1995): "
      f"PC1={pca_ref.explained_variance_ratio_[0]*100:.1f}%, "
      f"PC2={pca_ref.explained_variance_ratio_[1]*100:.1f}%")

for k in K_VALUES:
    print(f"\n{'='*40}")
    print(f"k = {k}")
    print(f"{'='*40}")
    for label, _ in TIME_VARIANTS:
        df = results[(k, label)]
        print(f"\n  --- {label} ({len(df)} countries) ---")
        for cid in sorted(df['Cluster'].unique()):
            sub = df[df['Cluster'] == cid]
            lbl = sub['ClusterLabels'].iloc[0]
            codes = ', '.join(sorted(sub['Country Code'].tolist()))
            print(f"    {lbl} (n={len(sub)}): {codes}")

print(f"\nSaved files:")
for k in K_VALUES:
    for label, _ in TIME_VARIANTS:
        print(f"  intermediary/clusters_k{k}_{label}.csv")
    print(f"  Graphics/NB4/biplot_k{k}_1995.html")
    print(f"  Graphics/NB4/map_k{k}_1995.html")


NB4: CLUSTERING SUMMARY

PCA variance explained (1995): PC1=41.3%, PC2=20.8%

k = 4

  --- 1995 (47 countries) ---
    Oil Exporters (n=16): AGO, ARG, AZE, BOL, CMR, COG, COL, ECU, EGY, EST, GAB, IRQ, MYS, NGA, TUN, YEM
    Hard Mineral Exporters (n=13): ALB, BIH, BWA, COD, DOM, GHA, GIN, JAM, MMR, MRT, PAK, VNM, ZWE
    Diversified Producers (n=10): CHL, IDN, KAZ, MEX, MNG, PER, ROU, RUS, UKR, ZAF
    Petrostates (n=8): BHR, DZA, IRN, KWT, OMN, QAT, SAU, VEN

  --- 2019 (47 countries) ---
    Oil Exporters (n=18): AGO, ALB, CMR, COG, COL, ECU, EGY, EST, GAB, GHA, MMR, NGA, PAK, ROU, TUN, UKR, VNM, YEM
    Diversified Producers (n=7): BOL, IDN, KAZ, MEX, MNG, PER, RUS
    Hard Mineral Exporters (n=10): BIH, BWA, CHL, COD, DOM, GIN, JAM, MRT, ZAF, ZWE
    Petrostates (n=12): ARG, AZE, BHR, DZA, IRN, IRQ, KWT, MYS, OMN, QAT, SAU, VEN

  --- agg (47 countries) ---
    Oil Exporters (n=16): AGO, ARG, AZE, BOL, CMR, COG, COL, ECU, EGY, EST, GAB, IRQ, MYS, NGA, TUN, YEM
    Hard Mineral Expo